In [110]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/internships.csv")

print(df.shape)
df.head()

(200, 17)


,internship_id,Date Time,profile,company,Location,Start Date,Stipend,Duration,Apply by Date,Skills,Eligibility,Perks,Description,deadline_is_demo,domain,work_mode,internship_type
0,2025WSHP0080,2025-09-21 15:24:28,Full Stack Developer,Cogent Web Services,Work from home,Immediately,"₹ 10,000 /month",3 Months,11 Oct' 25,"HTML, CSS, JavaScript, React.js, Node.js",Postgraduate,"Certificate, Letter Of Recommendation",This Full Stack Development internship is focu...,False,Web Development,Remote,Technical
1,2025WSHP0165,2025-09-21 15:58:52,UI/UX Design,Eduminatti,Work from home,Immediately,"₹ 5,000 - 10,000 /month",2 Months,18 Oct' 25,"Figma, UI Design, UX Design, Wireframing, Prot...",Postgraduate & Undergraduate,"Certificate, Letter Of Recommendation",This UI/UX Design internship is focused on des...,True,UI / UX,Remote,Technical
2,2025WSHP0242,2025-09-21 16:13:20,Frontend Developer,LSOYS Games & Apps,Work from home,Immediately,"₹ 1,000 /month",6 Months,20 Sep' 25,"HTML, CSS, JavaScript, React.js, TypeScript",Postgraduate & Undergraduate,"Certificate, Letter Of Recommendation",This Web Development internship is focused on ...,True,Web Development,Remote,Technical
3,2025WSHP0016,2025-09-27 23:11:11,Frontend Developer,InstaWeb Labs Private Limited,Mumbai,Immediately,"₹ 10,000 - 15,000 /month",6 Months,8 Oct' 25,"HTML, CSS, JavaScript, React.js, TypeScript",Postgraduate,"Certificate, Flexible, Letter Of Recommendatio...",This Front End Development internship is focus...,False,Web Development,Hybrid,Technical
4,2025WSHP0513,2025-09-21 16:02:53,Flutter Development,Pragament Tech Solutions Private Limited,Work from home,Immediately,"₹ 1,001 - 5,000 /month",1 Month,16 Sep' 25,"Dart, Flutter, Firebase, REST API, Git",Undergraduate,"Certificate, Flexible, Letter Of Recommendatio...",This Flutter Development internship is focused...,True,Software Development,Remote,Technical


In [111]:
# Rename columns to clean, consistent names

df = df.rename(columns={
    "Date Time": "posted_at",
    "profile": "role",
    "Location": "location",
    "Start Date": "start_date",
    "Stipend": "stipend",
    "Duration": "duration",
    "Apply by Date": "deadline",
    "Skills": "skills",
    "Eligibility": "eligibility",
    "Perks": "perks",
    "Description": "description"
})

# Convert all column names to lowercase
df.columns = df.columns.str.lower()

print(df.columns.tolist())

['internship_id', 'posted_at', 'role', 'company', 'location', 'start_date', 'stipend', 'duration', 'deadline', 'skills', 'eligibility', 'perks', 'description', 'deadline_is_demo', 'domain', 'work_mode', 'internship_type']


In [112]:
print(df.isnull().sum())

internship_id        0
posted_at            0
role                 0
company              0
location             0
start_date           0
stipend              0
duration             0
deadline             0
skills               0
eligibility          0
perks                0
description          0
deadline_is_demo    30
domain              30
work_mode            0
internship_type      0
dtype: int64


In [113]:
# Standardize common missing-value representations

missing_values = [
    "Not Available",
    "not available",
    "N/A",
    "n/a",
    "NA",
    "na",
    "Nothing",
    "nothing",
    ""
]

df = df.replace(missing_values, np.nan)

# Check missing values after standardization
print(df.isnull().sum())

internship_id        0
posted_at            0
role                 0
company              0
location             0
start_date           0
stipend              1
duration             0
deadline             2
skills               0
eligibility          0
perks                4
description          0
deadline_is_demo    30
domain              30
work_mode            0
internship_type      0
dtype: int64


In [114]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check duplicate internship IDs
print("Duplicate internship IDs:", df["internship_id"].duplicated().sum())

Duplicate rows: 0
Duplicate internship IDs: 0


In [115]:
# Clean text columns

text_columns = [
    "role",
    "company",
    "location",
    "skills",
    "eligibility",
    "perks",
    "description"
]

for col in text_columns:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print(df[text_columns].head())

                   role                                   company  \
0  Full Stack Developer                       Cogent Web Services   
1          UI/UX Design                                Eduminatti   
2    Frontend Developer                        LSOYS Games & Apps   
3    Frontend Developer             InstaWeb Labs Private Limited   
4   Flutter Development  Pragament Tech Solutions Private Limited   

         location                                             skills  \
0  Work from home           HTML, CSS, JavaScript, React.js, Node.js   
1  Work from home  Figma, UI Design, UX Design, Wireframing, Prot...   
2  Work from home        HTML, CSS, JavaScript, React.js, TypeScript   
3          Mumbai        HTML, CSS, JavaScript, React.js, TypeScript   
4  Work from home             Dart, Flutter, Firebase, REST API, Git   

                    eligibility  \
0                  Postgraduate   
1  Postgraduate & Undergraduate   
2  Postgraduate & Undergraduate   
3           

In [116]:
# Standardize location values
# Remote Work from home dono same meaning rakhte hain. Recommendation system ke liye inhe ek hi category banana better hai.

# Step 16 - Normalize location values

df["location"] = df["location"].replace({
    "Work from home": "Remote",
    "Remote": "Remote",
    "Bangalore": "Bengaluru"
})

print(df["location"].value_counts(dropna=False))

# Check unique values after standardization
print(df["location"].value_counts(dropna=False))

location
Remote           41
Chennai          28
Mumbai           23
Bengaluru        22
Delhi            21
Noida            19
Hyderabad        19
Pune             18
Thane             3
Greater Noida     3
Gurgaon           2
Dehradun          1
Name: count, dtype: int64
location
Remote           41
Chennai          28
Mumbai           23
Bengaluru        22
Delhi            21
Noida            19
Hyderabad        19
Pune             18
Thane             3
Greater Noida     3
Gurgaon           2
Dehradun          1
Name: count, dtype: int64


In [117]:
# handle stipend
import re

def extract_stipend(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])

    text = str(value).lower().strip()

    # Unpaid internship
    if "unpaid" in text:
        return pd.Series([0, 0])

    # Extract all numbers
    numbers = re.findall(r"\d[\d,]*", text)
    numbers = [int(num.replace(",", "")) for num in numbers]

    if len(numbers) >= 2:
        return pd.Series([numbers[0], numbers[1]])

    elif len(numbers) == 1:
        return pd.Series([numbers[0], numbers[0]])

    else:
        return pd.Series([np.nan, np.nan])


df[["stipend_min", "stipend_max"]] = df["stipend"].apply(
    extract_stipend
)

df[["stipend", "stipend_min", "stipend_max"]].head(20)


,stipend,stipend_min,stipend_max
0,"₹ 10,000 /month",10000.0,10000.0
1,"₹ 5,000 - 10,000 /month",5000.0,10000.0
2,"₹ 1,000 /month",1000.0,1000.0
3,"₹ 10,000 - 15,000 /month",10000.0,15000.0
4,"₹ 1,001 - 5,000 /month",1001.0,5000.0
5,"₹ 12,000 /month",12000.0,12000.0
6,"₹ 15,000 - 25,000 /month",15000.0,25000.0
7,"₹ 10,000 /month",10000.0,10000.0
8,"₹ 8,000 - 12,000 /month",8000.0,12000.0
9,"₹ 20,000 - 50,000 /month",20000.0,50000.0


In [118]:
# Calculate average stipend

df["stipend_avg"] = (
    df["stipend_min"] + df["stipend_max"]
) / 2

df[["stipend", "stipend_min", "stipend_max", "stipend_avg"]].head(20)

,stipend,stipend_min,stipend_max,stipend_avg
0,"₹ 10,000 /month",10000.0,10000.0,10000.0
1,"₹ 5,000 - 10,000 /month",5000.0,10000.0,7500.0
2,"₹ 1,000 /month",1000.0,1000.0,1000.0
3,"₹ 10,000 - 15,000 /month",10000.0,15000.0,12500.0
4,"₹ 1,001 - 5,000 /month",1001.0,5000.0,3000.5
5,"₹ 12,000 /month",12000.0,12000.0,12000.0
6,"₹ 15,000 - 25,000 /month",15000.0,25000.0,20000.0
7,"₹ 10,000 /month",10000.0,10000.0,10000.0
8,"₹ 8,000 - 12,000 /month",8000.0,12000.0,10000.0
9,"₹ 20,000 - 50,000 /month",20000.0,50000.0,35000.0


In [119]:
#Duration ko numeric me convert karna

def duration_to_months(value):
    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()

    # Weeks
    if "week" in text:
        numbers = re.findall(r"\d+(?:\.\d+)?", text)
        if numbers:
            return float(numbers[0]) / 4
        return np.nan

    # Months
    if "month" in text:
        numbers = re.findall(r"\d+(?:\.\d+)?", text)
        if numbers:
            return float(numbers[0])
        return np.nan

    return np.nan


df["duration_months"] = df["duration"].apply(duration_to_months)

df[["duration", "duration_months"]].drop_duplicates().sort_values(
    "duration_months"
)

,duration,duration_months
16,1 week ago,0.25
4,1 Month,1.00
1,2 Months,2.00
0,3 Months,3.00
37,4 Months,4.00
2,6 Months,6.00


In [120]:
# Convert skills string into a cleaned list

def clean_skills(value):
    if pd.isna(value):
        return []

    skills = str(value).split(",")

    return [
        skill.strip().lower()
        for skill in skills
        if skill.strip()
    ]


df["skills_list"] = df["skills"].apply(clean_skills)

# Check result
df[["skills", "skills_list"]].tail(10)

,skills,skills_list
190,"English Proficiency (Spoken), English Proficie...","[english proficiency (spoken), english profici..."
191,"Effective Communication, MS-Excel, MS-PowerPoi...","[effective communication, ms-excel, ms-powerpo..."
192,"English Proficiency (Spoken), English Proficie...","[english proficiency (spoken), english profici..."
193,"Content Editing, E-commerce, MS-Excel, Photogr...","[content editing, e-commerce, ms-excel, photog..."
194,"Business Management, Business Research, Conten...","[business management, business research, conte..."
195,"Content Writing, Creative Writing, English Pro...","[content writing, creative writing, english pr..."
196,"Canva, English Proficiency (Spoken), English P...","[canva, english proficiency (spoken), english ..."
197,"English Proficiency (Spoken), English Proficie...","[english proficiency (spoken), english profici..."
198,"Attention to Detail, Email Management, Google ...","[attention to detail, email management, google..."
199,"Artificial intelligence, Computer skills, Engl...","[artificial intelligence, computer skills, eng..."


In [121]:
df['skills_list'].head(20)

0            [html, css, javascript, react.js, node.js]
1     [figma, ui design, ux design, wireframing, pro...
2         [html, css, javascript, react.js, typescript]
3         [html, css, javascript, react.js, typescript]
4              [dart, flutter, firebase, rest api, git]
5          [python, node.js, express.js, rest api, sql]
6           [python, sql, pandas, power bi, statistics]
7         [html, css, javascript, react.js, typescript]
8     [python, pytorch, tensorflow, deep learning, nlp]
9     [kotlin, java, android studio, android sdk, fi...
10            [c++, java, python, oop, data structures]
11          [python, sql, pandas, power bi, statistics]
12                      [c, c++, arduino, esp32, stm32]
13         [python, linux, networking, nmap, wireshark]
14    [python, numpy, pandas, scikit-learn, machine ...
15         [python, llm, rag, embeddings, hugging face]
16    [selenium, postman, pytest, api testing, autom...
17            [c++, java, python, oop, data stru

In [122]:
# Standardize eligibility values

df["eligibility"] = (
    df["eligibility"]
    .astype("string")
    .str.strip()
)

# Check all unique eligibility values
print(df["eligibility"].value_counts(dropna=False))

eligibility
Undergraduate                   69
Postgraduate                    64
Undergraduate & Postgraduate    44
Postgraduate & Undergraduate    23
Name: count, dtype: Int64


In [123]:
# Standardize equivalent eligibility categories

df["eligibility"] = df["eligibility"].replace({
    "Undergraduate & Postgraduate": "Postgraduate & Undergraduate"
})

# Check again
print(df["eligibility"].value_counts(dropna=False))

eligibility
Undergraduate                   69
Postgraduate & Undergraduate    67
Postgraduate                    64
Name: count, dtype: Int64


In [124]:
# Convert date columns into datetime format

df["start_date"] = pd.to_datetime(
    df["start_date"],
    errors="coerce"
)

df["deadline"] = pd.to_datetime(
    df["deadline"],
    errors="coerce"
)

# Check the result
print("START DATE:")
print(df["start_date"].head(10))

print("\nDEADLINE:")
print(df["deadline"].head(10))

START DATE:
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: start_date, dtype: datetime64[s]

DEADLINE:
0   2025-10-11
1   2025-10-18
2   2025-09-20
3   2025-10-08
4   2025-09-16
5   2025-10-20
6   2025-10-16
7   2025-10-11
8   2025-10-16
9   2025-09-02
Name: deadline, dtype: datetime64[us]


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10756\151674490.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["start_date"] = pd.to_datetime(
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10756\151674490.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["deadline"] = pd.to_datetime(


In [125]:
print("Missing Start Dates:", df["start_date"].isna().sum())
print("Missing Deadlines:", df["deadline"].isna().sum())

Missing Start Dates: 200
Missing Deadlines: 3


In [126]:
# Reload raw dataset to inspect original date formats

raw_df = pd.read_csv("../data/internships.csv")

print("START DATE EXAMPLES:")
print(raw_df["Start Date"].head(20).to_string(index=False))

print("\nDEADLINE EXAMPLES:")
print(raw_df["Apply by Date"].head(20).to_string(index=False))

START DATE EXAMPLES:
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately
Immediately

DEADLINE EXAMPLES:
11 Oct' 25
18 Oct' 25
20 Sep' 25
 8 Oct' 25
16 Sep' 25
20 Oct' 25
16 Oct' 25
11 Oct' 25
16 Oct' 25
02 Sep' 25
10 Sep' 25
15 Oct' 25
31 Oct' 25
 5 Nov' 25
16 Oct' 25
 5 Nov' 25
 8 Oct' 25
 4 Oct' 25
13 Oct' 25
31 Oct' 25


In [127]:
print("\nUnique Start Date values:")
print(raw_df["Start Date"].dropna().unique()[:30])

print("\nUnique Deadline values:")
print(raw_df["Apply by Date"].dropna().unique()[:30])


Unique Start Date values:
<StringArray>
[                                                                      'Immediately',
 '2. can start the work from home job/internship between 1st Dec'25 and 31st Dec'25',
                                                                    'Within 30 days']
Length: 3, dtype: str

Unique Deadline values:
<StringArray>
[        '11 Oct' 25',         '18 Oct' 25',         '20 Sep' 25',
          '8 Oct' 25',         '16 Sep' 25',         '20 Oct' 25',
         '16 Oct' 25',         '02 Sep' 25',         '10 Sep' 25',
         '15 Oct' 25',         '31 Oct' 25',          '5 Nov' 25',
          '4 Oct' 25',         '13 Oct' 25',         '19 Oct' 25',
         '27 Oct' 25',         '26 Oct' 25',         '22 Oct' 25',
 '1 Dec - 31 Dec' 25',         '28 Nov' 25',         '03 Aug' 25',
         '24 Oct' 25',          '6 Oct' 25',         '23 Oct' 25',
         '20 Nov' 25',         '12 Oct' 25',         '26 Nov' 25',
         '20 Aug' 25',         '18 Se

In [128]:
# Reload raw date columns for preprocessing
raw_df = pd.read_csv("../data/internships.csv")

# Start type classification
def classify_start(value):
    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()

    if "immediately" in text:
        return "Immediately"

    if "within 30 days" in text:
        return "Within 30 days"

    if "between" in text or "and 31st" in text:
        return "Date Range"

    return "Other"


df["start_type"] = raw_df["Start Date"].apply(classify_start)

print(df["start_type"].value_counts(dropna=False))

start_type
Immediately       155
Within 30 days     44
Date Range          1
Name: count, dtype: int64


In [129]:
# Check the different deadline formats

print(df["deadline"].dropna().unique())

<DatetimeArray>
['2025-10-11 00:00:00', '2025-10-18 00:00:00', '2025-09-20 00:00:00',
 '2025-10-08 00:00:00', '2025-09-16 00:00:00', '2025-10-20 00:00:00',
 '2025-10-16 00:00:00', '2025-09-02 00:00:00', '2025-09-10 00:00:00',
 '2025-10-15 00:00:00',
 ...
 '2025-11-14 00:00:00', '2025-09-26 00:00:00', '2025-09-13 00:00:00',
 '2025-10-17 00:00:00', '2025-10-07 00:00:00', '2025-12-22 00:00:00',
 '2025-10-30 00:00:00', '2025-10-25 00:00:00', '2025-10-10 00:00:00',
 '2025-10-01 00:00:00']
Length: 103, dtype: datetime64[us]


In [130]:
print("Missing deadlines:", df["deadline"].isna().sum())

print("\nRows with missing deadline:")
print(
    df.loc[df["deadline"].isna(), ["internship_id", "deadline"]]
    .head(20)
)

Missing deadlines: 3

Rows with missing deadline:
    internship_id deadline
28   2025WSHP2348      NaT
183  2025WSHP0156      NaT
194  2025WSHP0222      NaT


In [131]:
# Read the original deadline values again
raw_df = pd.read_csv("../data/internships.csv")

raw_deadlines = raw_df["Apply by Date"]

print(raw_deadlines.dropna().unique())

<StringArray>
[   '11 Oct' 25',    '18 Oct' 25',    '20 Sep' 25',     '8 Oct' 25',
    '16 Sep' 25',    '20 Oct' 25',    '16 Oct' 25',    '02 Sep' 25',
    '10 Sep' 25',    '15 Oct' 25',
 ...
    '07 Oct' 25',    '22 Dec' 25',    '30 Oct' 25',    '25 Oct' 25',
     '2 Oct' 25',     '3 Oct' 25',     '9 Oct' 25', 'Not Available',
    '10 Oct' 25',     '1 Oct' 25']
Length: 112, dtype: str


In [132]:
'''def parse_deadline(value):
    if pd.isna(value):
        return pd.NaT

    text = str(value).strip()

    # Missing value
    if text.lower() == "not available":
        return pd.NaT

    # Case 1: Date range e.g. "1 Dec - 31 Dec' 25"
    if " - " in text:
        end_date = text.split(" - ")[-1].strip()

        # Convert "31 Dec' 25" → "31 Dec 2025"
        end_date = end_date.replace("'", "")
        return pd.to_datetime(end_date, errors="coerce")

    # Case 2: Normal date e.g. "11 Oct' 25"
    text = text.replace("'", "")
    return pd.to_datetime(text, errors="coerce")


df["deadline"] = raw_df["Apply by Date"].apply(parse_deadline)

print("Missing deadlines:", df["deadline"].isna().sum())

print("\nSample:")
print(df[["deadline"]].head(20))'''

'def parse_deadline(value):\n    if pd.isna(value):\n        return pd.NaT\n\n    text = str(value).strip()\n\n    # Missing value\n    if text.lower() == "not available":\n        return pd.NaT\n\n    # Case 1: Date range e.g. "1 Dec - 31 Dec\' 25"\n    if " - " in text:\n        end_date = text.split(" - ")[-1].strip()\n\n        # Convert "31 Dec\' 25" → "31 Dec 2025"\n        end_date = end_date.replace("\'", "")\n        return pd.to_datetime(end_date, errors="coerce")\n\n    # Case 2: Normal date e.g. "11 Oct\' 25"\n    text = text.replace("\'", "")\n    return pd.to_datetime(text, errors="coerce")\n\n\ndf["deadline"] = raw_df["Apply by Date"].apply(parse_deadline)\n\nprint("Missing deadlines:", df["deadline"].isna().sum())\n\nprint("\nSample:")\nprint(df[["deadline"]].head(20))'

In [133]:
# Robust deadline parser for the actual dataset format

import re
import pandas as pd
import numpy as np

def parse_deadline(value):
    if pd.isna(value):
        return pd.NaT

    text = str(value).strip()

    # Missing
    if text.lower() == "not available":
        return pd.NaT

    # If there is a range, take the END date
    if " - " in text:
        text = text.split(" - ")[-1].strip()

    # Convert:
    # "11 Oct' 25" -> "11 Oct 25"
    text = re.sub(r"'\s*", " ", text)

    # Remove any repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Parse explicitly as day-month-year
    return pd.to_datetime(
        text,
        format="%d %b %y",
        errors="coerce"
    )


# IMPORTANT: use RAW deadline values
raw_df = pd.read_csv("../data/internships.csv")

df["deadline"] = raw_df["Apply by Date"].apply(parse_deadline)

print("Missing deadlines:", df["deadline"].isna().sum())

print("\nParsed deadline values:")
print(df["deadline"].value_counts(dropna=False))

Missing deadlines: 2

Parsed deadline values:
deadline
2025-10-08    6
2025-10-15    6
2025-11-05    6
2025-10-26    6
2025-10-16    5
             ..
2025-09-13    1
2025-10-07    1
2025-12-22    1
2025-10-30    1
2025-10-01    1
Name: count, Length: 104, dtype: int64


In [134]:
# Check the original deadline values for rows
# that became NaT after parsing

check = pd.DataFrame({
    "raw_deadline": raw_df["Apply by Date"],
    "parsed_deadline": df["deadline"]
})

nat_rows = check[check["parsed_deadline"].isna()]

print(nat_rows["raw_deadline"].value_counts(dropna=False).head(30))

raw_deadline
Not Available    2
Name: count, dtype: int64


In [135]:
print("Total rows:", len(raw_df))
print("Raw deadline missing:", raw_df["Apply by Date"].isna().sum())

print("\nRaw deadline non-null values:")
print(raw_df["Apply by Date"].dropna().value_counts().head(30))

Total rows: 200
Raw deadline missing: 0

Raw deadline non-null values:
Apply by Date
15 Oct' 25    6
26 Oct' 25    6
8 Oct' 25     5
16 Oct' 25    5
5 Nov' 25     5
22 Oct' 25    5
12 Oct' 25    5
31 Oct' 25    4
27 Oct' 25    4
28 Nov' 25    4
23 Oct' 25    4
05 Oct' 25    4
11 Oct' 25    3
18 Oct' 25    3
20 Sep' 25    3
16 Sep' 25    3
19 Oct' 25    3
27 Aug' 25    3
06 Oct' 25    3
15 Sep' 25    3
23 Aug' 25    3
01 Nov' 25    3
17 Oct' 25    3
25 Oct' 25    3
02 Sep' 25    2
13 Oct' 25    2
24 Oct' 25    2
26 Nov' 25    2
28 Dec' 25    2
06 Nov' 25    2
Name: count, dtype: int64


In [136]:
# step - 14
# Clean internship descriptions

df["description"] = (
    df["description"]
    .astype("string")
    .str.strip()
)

print("Missing descriptions:", df["description"].isna().sum())

print("\nSample descriptions:")
print(df["description"].head(5).to_string(index=False))

Missing descriptions: 0

Sample descriptions:
This Full Stack Development internship is focus...
This UI/UX Design internship is focused on desi...
This Web Development internship is focused on s...
This Front End Development internship is focuse...
This Flutter Development internship is focused ...


In [137]:
#s-15
# Clean perks text

df["perks"] = (
    df["perks"]
    .astype("string")
    .str.strip()
)

print("Missing perks:", df["perks"].isna().sum())

print("\nPerks sample:")
print(df["perks"].head(10).to_string(index=False))

Missing perks: 4

Perks sample:
             Certificate, Letter Of Recommendation
             Certificate, Letter Of Recommendation
             Certificate, Letter Of Recommendation
Certificate, Flexible, Letter Of Recommendation...
Certificate, Flexible, Letter Of Recommendation...
                                              <NA>
Certificate, Flexible, Letter Of Recommendation...
             Certificate, Letter Of Recommendation
                                       Certificate
   Certificate, Flexible, Letter Of Recommendation


In [138]:
# Final validation

print("Dataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isna().sum())

# Duplicate check using original/hashable columns
duplicate_columns = [
    col for col in df.columns
    if col != "skills_list"
]

print("\nDuplicate Rows:")
print(df.duplicated(subset=duplicate_columns).sum())

print("\nDuplicate Internship IDs:")
print(df["internship_id"].duplicated().sum())

Dataset Shape:
(200, 23)

Missing Values:
internship_id         0
posted_at             0
role                  0
company               0
location              0
start_date          200
stipend               1
duration              0
deadline              2
skills                0
eligibility           0
perks                 4
description           0
deadline_is_demo     30
domain               30
work_mode             0
internship_type       0
stipend_min           1
stipend_max           1
stipend_avg           1
duration_months       0
skills_list           0
start_type            0
dtype: int64

Duplicate Rows:
0

Duplicate Internship IDs:
0


In [139]:
df

,internship_id,posted_at,role,company,location,start_date,stipend,duration,deadline,skills,...,deadline_is_demo,domain,work_mode,internship_type,stipend_min,stipend_max,stipend_avg,duration_months,skills_list,start_type
0,2025WSHP0080,2025-09-21 15:24:28,Full Stack Developer,Cogent Web Services,Remote,NaT,"₹ 10,000 /month",3 Months,2025-10-11,"HTML, CSS, JavaScript, React.js, Node.js",...,False,Web Development,Remote,Technical,10000.0,10000.0,10000.0,3.0,"[html, css, javascript, react.js, node.js]",Immediately
1,2025WSHP0165,2025-09-21 15:58:52,UI/UX Design,Eduminatti,Remote,NaT,"₹ 5,000 - 10,000 /month",2 Months,2025-10-18,"Figma, UI Design, UX Design, Wireframing, Prot...",...,True,UI / UX,Remote,Technical,5000.0,10000.0,7500.0,2.0,"[figma, ui design, ux design, wireframing, pro...",Immediately
2,2025WSHP0242,2025-09-21 16:13:20,Frontend Developer,LSOYS Games & Apps,Remote,NaT,"₹ 1,000 /month",6 Months,2025-09-20,"HTML, CSS, JavaScript, React.js, TypeScript",...,True,Web Development,Remote,Technical,1000.0,1000.0,1000.0,6.0,"[html, css, javascript, react.js, typescript]",Immediately
3,2025WSHP0016,2025-09-27 23:11:11,Frontend Developer,InstaWeb Labs Private Limited,Mumbai,NaT,"₹ 10,000 - 15,000 /month",6 Months,2025-10-08,"HTML, CSS, JavaScript, React.js, TypeScript",...,False,Web Development,Hybrid,Technical,10000.0,15000.0,12500.0,6.0,"[html, css, javascript, react.js, typescript]",Immediately
4,2025WSHP0513,2025-09-21 16:02:53,Flutter Development,Pragament Tech Solutions Private Limited,Remote,NaT,"₹ 1,001 - 5,000 /month",1 Month,2025-09-16,"Dart, Flutter, Firebase, REST API, Git",...,True,Software Development,Remote,Technical,1001.0,5000.0,3000.5,1.0,"[dart, flutter, firebase, rest api, git]",Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2025WSHP0603,2025-09-27 22:39:04,Media & Public Relations (PR),Veritas Reputation PR Private Limited,Mumbai,NaT,"₹ 5,000 - 10,000 /month",2 Months,2025-10-22,"Content Writing, Creative Writing, English Pro...",...,NaN,NaN,On-site,Non-Technical,5000.0,10000.0,7500.0,2.0,"[content writing, creative writing, english pr...",Immediately
196,2025WSHP0202,2025-09-27 21:47:17,Market Research,Suspol Technologies Private Limited,Greater Noida,NaT,"₹ 7,000 - 10,000 /month",3 Months,2025-10-08,"Canva, English Proficiency (Spoken), English P...",...,NaN,NaN,On-site,Research,7000.0,10000.0,8500.0,3.0,"[canva, english proficiency (spoken), english ...",Immediately
197,2025WSHP1518,2025-09-27 20:54:31,Business Research,SCDND ESTATES PRIVATE LIMITED,Greater Noida,NaT,"₹ 9,000 - 13,000 /month",3 Months,2025-10-18,"English Proficiency (Spoken), English Proficie...",...,NaN,NaN,On-site,Research,9000.0,13000.0,11000.0,3.0,"[english proficiency (spoken), english profici...",Immediately
198,2025WSHP0769,2025-09-27 21:11:47,Research & Insights,Caarya,Chennai,NaT,"₹ 7,000 - 15,000 /month",2 Months,2025-10-22,"Attention to Detail, Email Management, Google ...",...,NaN,NaN,On-site,Research,7000.0,15000.0,11000.0,2.0,"[attention to detail, email management, google...",Immediately


In [140]:
# Remove the incorrectly parsed start_date column
df = df.drop(columns=["start_date"])

# We already have the useful representation in start_type.
# Keep original start information separately if needed.
raw_df = pd.read_csv("../data/internships.csv")

df["start_date_raw"] = raw_df["Start Date"]

print(df[["start_date_raw", "start_type"]].head(10))

  start_date_raw   start_type
0    Immediately  Immediately
1    Immediately  Immediately
2    Immediately  Immediately
3    Immediately  Immediately
4    Immediately  Immediately
5    Immediately  Immediately
6    Immediately  Immediately
7    Immediately  Immediately
8    Immediately  Immediately
9    Immediately  Immediately


In [141]:
# Final validation

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isna().sum())

print("\nDuplicate Rows:")
print(
    df.duplicated(
        subset=[col for col in df.columns if col != "skills_list"]
    ).sum()
)

print("\nDuplicate Internship IDs:")
print(df["internship_id"].duplicated().sum())

Dataset Shape:
(200, 23)

Columns:
['internship_id', 'posted_at', 'role', 'company', 'location', 'stipend', 'duration', 'deadline', 'skills', 'eligibility', 'perks', 'description', 'deadline_is_demo', 'domain', 'work_mode', 'internship_type', 'stipend_min', 'stipend_max', 'stipend_avg', 'duration_months', 'skills_list', 'start_type', 'start_date_raw']

Missing Values:
internship_id        0
posted_at            0
role                 0
company              0
location             0
stipend              1
duration             0
deadline             2
skills               0
eligibility          0
perks                4
description          0
deadline_is_demo    30
domain              30
work_mode            0
internship_type      0
stipend_min          1
stipend_max          1
stipend_avg          1
duration_months      0
skills_list          0
start_type           0
start_date_raw       0
dtype: int64

Duplicate Rows:
0

Duplicate Internship IDs:
0


In [142]:
# Step 16 - Add Work Mode to current preprocessing dataframe

raw_workmode_df = pd.read_csv("../data/internships.csv")

# Copy work_mode from the current internship dataset
df["work_mode"] = raw_workmode_df["work_mode"]

print("Work mode added:", "work_mode" in df.columns)

print("\nWork Mode Distribution:")
print(df["work_mode"].value_counts(dropna=False))

Work mode added: True

Work Mode Distribution:
work_mode
On-site    106
Hybrid      53
Remote      41
Name: count, dtype: int64


In [143]:
# Step 16 - Verify Work Mode and Domain

print(df.columns.tolist())

print("\nDomain:")
print(df["domain"].value_counts())

print("\nWork Mode:")
print(df["work_mode"].value_counts())

['internship_id', 'posted_at', 'role', 'company', 'location', 'stipend', 'duration', 'deadline', 'skills', 'eligibility', 'perks', 'description', 'deadline_is_demo', 'domain', 'work_mode', 'internship_type', 'stipend_min', 'stipend_max', 'stipend_avg', 'duration_months', 'skills_list', 'start_type', 'start_date_raw']

Domain:
domain
Software Development     66
AI / Machine Learning    32
Web Development          25
Data Science             23
Cloud Computing          12
UI / UX                   6
Cyber Security            6
Name: count, dtype: int64

Work Mode:
work_mode
On-site    106
Hybrid      53
Remote      41
Name: count, dtype: int64


In [144]:
# Check before saving

print("internship_type" in df.columns)
print(df["internship_type"].value_counts())

True
internship_type
Technical        170
Non-Technical     26
Research           4
Name: count, dtype: int64


In [145]:
# Save the processed dataset

output_path = "../data/processed_internships.csv"

df.to_csv(output_path, index=False)

print(f"Processed dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")

#verify

check_df = pd.read_csv(output_path)

print(check_df.shape)
print(check_df.head())

Processed dataset saved to: ../data/processed_internships.csv
Final shape: (200, 23)
(200, 23)
  internship_id            posted_at                  role  \
0  2025WSHP0080  2025-09-21 15:24:28  Full Stack Developer   
1  2025WSHP0165  2025-09-21 15:58:52          UI/UX Design   
2  2025WSHP0242  2025-09-21 16:13:20    Frontend Developer   
3  2025WSHP0016  2025-09-27 23:11:11    Frontend Developer   
4  2025WSHP0513  2025-09-21 16:02:53   Flutter Development   

                                    company location  \
0                       Cogent Web Services   Remote   
1                                Eduminatti   Remote   
2                        LSOYS Games & Apps   Remote   
3             InstaWeb Labs Private Limited   Mumbai   
4  Pragament Tech Solutions Private Limited   Remote   

                    stipend  duration    deadline  \
0           ₹ 10,000 /month  3 Months  2025-10-11   
1   ₹ 5,000 - 10,000 /month  2 Months  2025-10-18   
2            ₹ 1,000 /month  6 Month

In [146]:
df

,internship_id,posted_at,role,company,location,stipend,duration,deadline,skills,eligibility,...,domain,work_mode,internship_type,stipend_min,stipend_max,stipend_avg,duration_months,skills_list,start_type,start_date_raw
0,2025WSHP0080,2025-09-21 15:24:28,Full Stack Developer,Cogent Web Services,Remote,"₹ 10,000 /month",3 Months,2025-10-11,"HTML, CSS, JavaScript, React.js, Node.js",Postgraduate,...,Web Development,Remote,Technical,10000.0,10000.0,10000.0,3.0,"[html, css, javascript, react.js, node.js]",Immediately,Immediately
1,2025WSHP0165,2025-09-21 15:58:52,UI/UX Design,Eduminatti,Remote,"₹ 5,000 - 10,000 /month",2 Months,2025-10-18,"Figma, UI Design, UX Design, Wireframing, Prot...",Postgraduate & Undergraduate,...,UI / UX,Remote,Technical,5000.0,10000.0,7500.0,2.0,"[figma, ui design, ux design, wireframing, pro...",Immediately,Immediately
2,2025WSHP0242,2025-09-21 16:13:20,Frontend Developer,LSOYS Games & Apps,Remote,"₹ 1,000 /month",6 Months,2025-09-20,"HTML, CSS, JavaScript, React.js, TypeScript",Postgraduate & Undergraduate,...,Web Development,Remote,Technical,1000.0,1000.0,1000.0,6.0,"[html, css, javascript, react.js, typescript]",Immediately,Immediately
3,2025WSHP0016,2025-09-27 23:11:11,Frontend Developer,InstaWeb Labs Private Limited,Mumbai,"₹ 10,000 - 15,000 /month",6 Months,2025-10-08,"HTML, CSS, JavaScript, React.js, TypeScript",Postgraduate,...,Web Development,Hybrid,Technical,10000.0,15000.0,12500.0,6.0,"[html, css, javascript, react.js, typescript]",Immediately,Immediately
4,2025WSHP0513,2025-09-21 16:02:53,Flutter Development,Pragament Tech Solutions Private Limited,Remote,"₹ 1,001 - 5,000 /month",1 Month,2025-09-16,"Dart, Flutter, Firebase, REST API, Git",Undergraduate,...,Software Development,Remote,Technical,1001.0,5000.0,3000.5,1.0,"[dart, flutter, firebase, rest api, git]",Immediately,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2025WSHP0603,2025-09-27 22:39:04,Media & Public Relations (PR),Veritas Reputation PR Private Limited,Mumbai,"₹ 5,000 - 10,000 /month",2 Months,2025-10-22,"Content Writing, Creative Writing, English Pro...",Postgraduate,...,NaN,On-site,Non-Technical,5000.0,10000.0,7500.0,2.0,"[content writing, creative writing, english pr...",Immediately,Immediately
196,2025WSHP0202,2025-09-27 21:47:17,Market Research,Suspol Technologies Private Limited,Greater Noida,"₹ 7,000 - 10,000 /month",3 Months,2025-10-08,"Canva, English Proficiency (Spoken), English P...",Postgraduate & Undergraduate,...,NaN,On-site,Research,7000.0,10000.0,8500.0,3.0,"[canva, english proficiency (spoken), english ...",Immediately,Immediately
197,2025WSHP1518,2025-09-27 20:54:31,Business Research,SCDND ESTATES PRIVATE LIMITED,Greater Noida,"₹ 9,000 - 13,000 /month",3 Months,2025-10-18,"English Proficiency (Spoken), English Proficie...",Undergraduate,...,NaN,On-site,Research,9000.0,13000.0,11000.0,3.0,"[english proficiency (spoken), english profici...",Immediately,Immediately
198,2025WSHP0769,2025-09-27 21:11:47,Research & Insights,Caarya,Chennai,"₹ 7,000 - 15,000 /month",2 Months,2025-10-22,"Attention to Detail, Email Management, Google ...",Postgraduate,...,NaN,On-site,Research,7000.0,15000.0,11000.0,2.0,"[attention to detail, email management, google...",Immediately,Immediately
